## 1. Overview

End-to-end pipeline transforming raw CDC data into a historized SCD Type 2 model,
exposed through a Snowflake Semantic View and queryable in natural language via a
Cortex Agent.

**Schemas inside `GDS_HA_MARC_MEDAWAR`:**
- `STAGING`  deduped, normalized views over raw
- `CURATED`  SCD2 tables and product dimension
- `SEMANTIC`  Snowflake Semantic View + Cortex Agent

## 2. Data model

| Object | Grain | Keys |
|---|---|---|
| `CURATED.DIM_CUSTOMER_SCD2` | one row per (customer_id, valid_from) | unique: `CUSTOMER_ID` (logical) |
| `CURATED.FCT_CUSTOMER_PRODUCT_HOLDING_SCD2` | one row per (holding_id, valid_from) |  PK: `HOLDING_ID` |
| `CURATED.DIM_PRODUCT` | one row per product | PK: `PRODUCT_ID` |

SCD2 tables have `VALID_FROM`, `VALID_TO`, `IS_CURRENT`.
`DIM_PRODUCT` has derived `IS_MORTGAGE`, `IS_PNC_INSURANCE` flags.

**Relationships (many-to-one):**
- `FCT_..._HOLDING_SCD2.CUSTOMER_ID` → `DIM_CUSTOMER_SCD2.CUSTOMER_ID`
- `FCT_..._HOLDING_SCD2.PRODUCT_ID` → `DIM_PRODUCT.PRODUCT_ID`

**Metric:** `TOTAL_HOLDING_AMOUNT = SUM(AMOUNT)` in NOK. Means different things by
category (loan balance for mortgages, premium for insurance); aggregates are most
meaningful when scoped to a single product category.

## 3. CDC → SCD2 conversion

**Dedupe (staging):** replay duplicates handled by keeping the latest `ingested_at`
per `source_event_id`. Codes normalized with `UPPER(TRIM(...))` to neutralize raw
formatting noise (`" pnc "`, `" mm "`, `" act "`).

**SCD2 versioning (curated):** each CDC event becomes one version.

```sql
valid_from = event_datetime
valid_to   = COALESCE(
    LEAD(event_datetime) OVER (PARTITION BY entity_id ORDER BY event_datetime),
    TIMESTAMP '9999-12-31 00:00:00'
)
is_current = (valid_to = TIMESTAMP '9999-12-31 00:00:00')
```

**Delete handling:** D events are filtered from the final SCD2 (`WHERE cdc_operation != 'D'`).
Their effect is captured implicitly, the LEAD window on the prior version ends that
version's validity at the D timestamp. After a delete, the entity has no current row.


## 4. Semantic view & Cortex Agent

**Semantic view:** `GDS_HA_MARC_MEDAWAR.SEMANTIC.SV_CUSTOMER_PRODUCT_HOLDINGS`
Built via Snowsight UI. Exposes 3 tables, 2 relationships, 1 metric.

**Cortex Agent:** `GDS_HA_MARC_MEDAWAR.SEMANTIC.[AGENT_NAME]`
Tool: Cortex Analyst on the semantic view above.
Custom instructions encode SCD2 point-in-time logic and metric scoping.

**How to use:** Snowsight → AI & ML → Agents → [AGENT_NAME] → Chat.

## 5. Validation results (as of 2025-12-31)

| # | Prompt | Answer |
|---|---|---|
| 1 | Which customers have a mortgage but no P&C insurance as of 2025-12-31? | 19 customers (list returned) |
| 2 | How many customers have a mortgage as of 2025-12-31? | 50 |
| 3 | How many customers have P&C insurance as of 2025-12-31? | 64 |
| 4 | What is the total mortgage exposure as of 2025-12-31? | 188,746,000 NOK (~188.7M) |
| 5 | How many customers in each region have a mortgage as of 2025-12-31? | Oslo leads with 12 mortgage customers, followed by Troms and Agder with 7 each. The total across all 8 regions is 50 customers, |

All answers cross-checked with direct SQL queries against the curated SCD2 tables.


#  Home Assignment, SCD2 Pipeline + Semantic Layer + Cortex Agent


## Objective
transform the raw data into a historized SCD Type 2 model and expose a semantic layer that can be
used by a Snowflake Cortex Agent to answer business questions in natural language

### EDA 
Understand the data, the shape, spot duplicates and unclean data, identify mortgage and P&C products.

**Findings encoded into the pipeline:**
- `CUSTOMER_CDC_RAW` — `segment_code` and `customer_status_code` have mixed case and whitespace (e.g. `" afl "` vs `AFL`, `" act "` vs `ACT`)
- `CUSTOMER_PRODUCT_HOLDING_CDC_RAW` — `currency_code` and `product_id` have casing variants (`'NOK'` vs `'nok'`, `'p002'` vs `'P002'`)
- `PRODUCT_MASTER_RAW` — `product_group_code` and `product_category_code` have whitespace/case variants (`'ins'` vs `'INS'`, `'pnc'` vs `'PNC'`)
- Replay duplicates in both CDC tables share the same `source_event_id` with different `ingested_at` (broker-level retries, typically 30–45 min apart)

All of the above are handled by `UPPER(TRIM(...))` + dedupe in the staging layer.

In [ ]:
%%sql -r dataframe_2
SELECT * FROM GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_CDC_RAW ;
-- segmendt_ocde should be upper ( afl, AFL )
-- status code same act ACT

In [ ]:
%%sql -r dataframe_3
SELECT * FROM GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_PRODUCT_HOLDING_CDC_RAW ;
-- when CDC_operation D amount become null
-- currency 'NOK' VS 'nok'
-- product_id 'p002' VS 'P002'


In [ ]:
%%sql -r dataframe_4
SELECT * FROM GDS_HOME_ASSIGNMENT.RAW.PRODUCT_MASTER_RAW;
-- product_group_code 'ins' vs 'INS'
-- product_category_code 'pnc' vs 'PNC'

In [ ]:
%%sql -r dataframe_1
select source_event_id, count(*)  from GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_CDC_RAW
group by 1 having count(*) > 1;

In [ ]:
%%sql -r dataframe_13
-- dups we will check ingested at to determine which record to take
select * from GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_CDC_RAW where source_event_id in ( 'CRM-EVT-00109', 'CRM-EVT-00067');

In [ ]:
%%sql -r dataframe_5
SELECT source_event_id, COUNT(*) 
FROM GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_PRODUCT_HOLDING_CDC_RAW 
group by 1
having count(*) > 1

In [ ]:
%%sql -r dataframe_14
-- dups we will check ingested at to determine which record to take
select * from GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_PRODUCT_HOLDING_CDC_RAW where source_event_id in ( 'HLD-EVT-00071', 'HLD-EVT-00151');

 ### create schemas in GDS_HA_MARC_MEDAWAR (STAGING, CURATED, SEMANTIC )
 Everything is built inside `GDS_HA_MARC_MEDAWAR`
 
 STAGING schema: VIEWS deduped, cleaned over raw
 
 CURATED schema: DIM_CUSTOMER_SCD2, FCT_CUSTOMER_PRODUCT_HOLDING_SCD2, DIM_PRODUCT
 
 SEMANTIC schema: SEMANTIC VIEW object + the Cortex agent

In [ ]:
%%sql -r dataframe_6
USE DATABASE GDS_HA_MARC_MEDAWAR;

CREATE SCHEMA IF NOT EXISTS STAGING;
CREATE SCHEMA IF NOT EXISTS CURATED;
CREATE SCHEMA IF NOT EXISTS SEMANTIC;

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE VIEW GDS_HA_MARC_MEDAWAR.STAGING.STG_CUSTOMER AS
SELECT
    source_event_id,
    customer_id,
    source_system,
    national_id,
    trim(first_name) AS first_name,
    trim(last_name) AS last_name,
    birth_date,
    UPPER(trim(region_code)) AS region_code,
    trim(region_name) AS region_name,
    UPPER(trim(segment_code)) AS segment_code,
    UPPER(trim(customer_status_code)) AS customer_status_code,
    event_datetime,
    UPPER(trim(cdc_operation)) AS cdc_operation,
    ingested_at
FROM GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_CDC_RAW
QUALIFY row_number() OVER (
    PARTITION BY source_event_id
    order by ingested_at desc
) = 1;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE VIEW GDS_HA_MARC_MEDAWAR.STAGING.STG_HOLDING AS
SELECT
    source_event_id,
    holding_id,
    source_system,
    customer_id,
    product_id,
    amount,
    UPPER(TRIM(currency_code)) AS currency_code,
    event_datetime,
    UPPER(TRIM(cdc_operation)) AS cdc_operation,
    ingested_at
FROM GDS_HOME_ASSIGNMENT.RAW.CUSTOMER_PRODUCT_HOLDING_CDC_RAW
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY source_event_id
    ORDER BY ingested_at DESC
) = 1;

In [ ]:
%%sql -r dataframe_15
CREATE OR REPLACE VIEW GDS_HA_MARC_MEDAWAR.STAGING.STG_PRODUCT AS
SELECT
    product_id,
    source_system,
    TRIM(product_name) AS product_name,
    UPPER(TRIM(product_group_code)) AS product_group_code,
    TRIM(product_group_name) AS product_group_name,
    UPPER(TRIM(product_category_code)) AS product_category_code,
    TRIM(product_category_name) AS product_category_name,
    UPPER(TRIM(product_type_code)) AS product_type_code,
    TRIM(product_type_name) AS product_type_name
FROM GDS_HOME_ASSIGNMENT.RAW.PRODUCT_MASTER_RAW;

In [ ]:
%%sql -r dataframe_16
-- dups has disapeard
select source_event_id, count(*)  from GDS_HA_MARC_MEDAWAR.STAGING.STG_CUSTOMER
group by 1 having count(*) > 1;

In [ ]:
%%sql -r dataframe_17
SELECT p.product_category_name, 
       MIN(h.amount) AS min_amt, 
       AVG(h.amount) AS avg_amt, 
       MAX(h.amount) AS max_amt,
       COUNT(*) AS n
FROM GDS_HA_MARC_MEDAWAR.STAGING.STG_HOLDING h
JOIN GDS_HA_MARC_MEDAWAR.STAGING.STG_PRODUCT p USING (product_id)
GROUP BY p.product_category_name
ORDER BY avg_amt DESC;

## Curated models 

**Materialized as a table** (small, queried by downstream models and the semantic view).

**DIM_PRODUCT** one row per product clean codes, derived flags ( Mortgage flag — true if the customer holds at least one product categorized as a mortgage , P&C insurance flag, true if the customer holds at least one product categorized as property and casualty insurance)

**DIM_CUSTOMER_SCD2** one row per (customer_id, valid_from) SCD2 with `valid_from`, `valid_to`, `is_current`

**FCT_CUSTOMER_PRODUCT_HOLDING_SCD2** one row per (holding_id, valid_from) SCD2 historized at holding grain



In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE GDS_HA_MARC_MEDAWAR.CURATED.DIM_PRODUCT AS
SELECT
    product_id,
    source_system,
    product_name,
    product_group_code,
    product_group_name,
    product_category_code,
    product_category_name,
    product_type_code,
    product_type_name,
    (product_category_code = 'MORT') AS is_mortgage,
    (product_category_code = 'PNC')  AS is_pnc_insurance
FROM GDS_HA_MARC_MEDAWAR.STAGING.STG_PRODUCT;

### `DIM_CUSTOMER_SCD2` — customer SCD Type 2

Each row = one version of a customer over a time window.

- `valid_from` = `event_datetime` of the CDC event creating this version
- `valid_to`   = `event_datetime` of the next event for the same customer, or `9999-12-31` if no next event
- D events terminate the prior version (their timestamp becomes the prior row's `valid_to` via `LEAD`) and the D row itself is filtered out, so a deleted customer has no `is_current = TRUE` row and disappears from point-in-time queries after the delete
- `is_current = TRUE` iff `valid_to = 9999-12-31`

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE TABLE GDS_HA_MARC_MEDAWAR.CURATED.DIM_CUSTOMER_SCD2 AS
WITH versioned AS (
    SELECT
        customer_id,
        source_system,
        national_id,
        first_name,
        last_name,
        birth_date,
        region_code,
        region_name,
        segment_code,
        customer_status_code,
        cdc_operation,
        event_datetime AS valid_from,
        COALESCE(
            LEAD(event_datetime) OVER (
                PARTITION BY customer_id
                ORDER BY event_datetime
            ),
            TIMESTAMP '9999-12-31 00:00:00'
        ) AS valid_to
    FROM GDS_HA_MARC_MEDAWAR.STAGING.STG_CUSTOMER
)
SELECT
    customer_id,
    source_system,
    national_id,
    first_name,
    last_name,
    birth_date,
    region_code,
    region_name,
    segment_code,
    customer_status_code,
    valid_from,
    valid_to,
    (valid_to = TIMESTAMP '9999-12-31 00:00:00') AS is_current
FROM versioned
WHERE cdc_operation != 'D';  -- drop terminator rows; their effect is captured by LEAD on prior row

In [ ]:
%%sql -r dataframe_18
SELECT COUNT(*) AS scd2_rows,
       COUNT(DISTINCT customer_id) AS distinct_customers,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows
FROM GDS_HA_MARC_MEDAWAR.CURATED.DIM_CUSTOMER_SCD2;

### `FCT_CUSTOMER_PRODUCT_HOLDING_SCD2` holdings SCD Type 2

Same pattern as the customer SCD2, but **grain = `holding_id`**.


In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE TABLE GDS_HA_MARC_MEDAWAR.CURATED.FCT_CUSTOMER_PRODUCT_HOLDING_SCD2 AS
WITH versioned AS (
    SELECT
        holding_id,
        source_system,
        customer_id,
        product_id,
        amount,
        currency_code,
        cdc_operation,
        event_datetime AS valid_from,
        COALESCE(
            LEAD(event_datetime) OVER (
                PARTITION BY holding_id
                ORDER BY event_datetime
            ),
            TIMESTAMP '9999-12-31 00:00:00'
        ) AS valid_to
    FROM GDS_HA_MARC_MEDAWAR.STAGING.STG_HOLDING
)
SELECT
    holding_id,
    source_system,
    customer_id,
    product_id,
    amount,
    currency_code,
    valid_from,
    valid_to,
    (valid_to = TIMESTAMP '9999-12-31 00:00:00') AS is_current
FROM versioned
WHERE cdc_operation != 'D';

---

## Semantic view & Cortex Agent

Both objects were created via the Snowsight UI rather than in this notebook, to leverage the visual builder and automatic YAML generation used by Cortex Analyst.

**Semantic view:** `GDS_HA_MARC_MEDAWAR.SEMANTIC.SV_CUSTOMER_PRODUCT_HOLDINGS`

- 3 logical tables: `DIM_CUSTOMER_SCD2`, `DIM_PRODUCT`, `FCT_CUSTOMER_PRODUCT_HOLDING_SCD2`
- 2 many-to-one relationships: holding → customer, holding → product
- 1 metric: `TOTAL_HOLDING_AMOUNT = SUM(AMOUNT)` in NOK

**Cortex Agent:** `GDS_HA_MARC_MEDAWAR.SEMANTIC.AGT_CUSTOMER_PRODUCT_HOLDINGS`

- Tool: Cortex Analyst on the semantic view above
- Custom instructions encode SCD2 point-in-time logic (filter `VALID_FROM <= X AND X < VALID_TO`), mortgage/P&C flag usage, and metric scoping by product category

Both objects are visible in Snowsight under `AI & ML → Agents` and `AI & ML → Cortex Analyst` (or the database tree under the `SEMANTIC` schema).